In [ ]:
!pip3 install -U ucimlrepo 

https://github.com/uci-ml-repo/ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo, list_available_datasets
list_available_datasets()

In [ ]:
import pandas as pd

handwriting = fetch_ucirepo(id = 80)
handwriting

In [35]:
test = handwriting.data
for i in test:
    print (i)

ids
features
targets
original
headers


In [36]:
df = pd.DataFrame(handwriting['data']['features'])

df

,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,...,Attribute55,Attribute56,Attribute57,Attribute58,Attribute59,Attribute60,Attribute61,Attribute62,Attribute63,Attribute64
0,0,1,6,15,12,1,0,0,0,7,...,0,0,0,0,6,14,7,1,0,0
1,0,0,10,16,6,0,0,0,0,7,...,3,0,0,0,10,16,15,3,0,0
2,0,0,8,15,16,13,0,0,0,1,...,0,0,0,0,9,14,0,0,0,0
3,0,0,0,3,11,16,0,0,0,0,...,0,0,0,0,0,1,15,2,0,0
4,0,0,5,14,4,0,0,0,0,0,...,12,0,0,0,4,12,14,7,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5615,0,0,4,10,13,6,0,0,0,1,...,4,0,0,0,2,14,15,9,0,0
5616,0,0,6,16,13,11,1,0,0,0,...,1,0,0,0,6,16,14,6,0,0
5617,0,0,1,11,15,1,0,0,0,0,...,0,0,0,0,2,9,13,6,0,0
5618,0,0,2,10,7,0,0,0,0,0,...,2,0,0,0,5,12,16,12,0,0


In [37]:
df = handwriting.data['targets']
df

,class
0,0
1,0
2,7
3,4
4,6
...,...
5615,9
5616,0
5617,8
5618,9


In [38]:
%pip install -q scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [67]:
# Since this is a MVP, we just use the MVP itself first!
# Now, we need to do train and split. 

from sklearn.model_selection import train_test_split

X = handwriting.data.features
y = handwriting.data.targets.iloc[:, 0]

X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.7, random_state = 42, stratify=y)

In [79]:
# Because KNN is a lazy learner (doesn't have a generalized equation or anything)
# We need to find ideal k values, and we want to randomize 
# Best method is using CV
# Library used: GridSearch CV

from sklearn.model_selection import GridSearchCV, KFold
from sklearn.neighbors import KNeighborsClassifier

folds = KFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid={"n_neighbors": range(1, 30)},
    cv=folds,
)

search.fit(X_train, y_train)

print(search.best_params_)

{'n_neighbors': 6}


In [80]:
# Since best params is at k = 6, we can finally store the model.
y_pred = search.predict(X_test)

# using this simple accuracy method
def testing(y_pred, y_truth):
    count = 0
    total = 0

    for var1, var2 in zip(y_pred, y_truth):
        total += 1
        if var1 == var2:
            count += 1

    return (count / total)*100

test_accuracy = testing(y_pred, y_test)

# Best among cross validation
print("Cross-validation accuracy:", search.best_score_)

# Test for the dataset
print("Test accuracy:", test_accuracy)

Cross-validation accuracy: 0.9857600124154923
Test accuracy: 98.1624184943687


In [ ]:
# Test for accuracy
# Use f1 score.
from sklearn.metrics import f1_score
score = f1_score(y_test, y_pred, average="weighted")
print(score*100)

# Result: Pretty accurate.

98.15936061508816
